In [ ]:
!pip install mediapipe
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 20.6 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import glob
import subprocess
import numpy as np
import librosa
import mediapipe as mp
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

zip_path = '/content/drive/My Drive/FakeAVCeleb_v1.2.zip'
extract_path = '/content/FakeAVCeleb'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dosyalar şu klasöre çıkarıldı: {extract_path}")

Dosyalar şu klasöre çıkarıldı: /content/FakeAVCeleb


In [ ]:
DATASET_DIR = "/content/FakeAVCeleb/FakeAVCeleb_v1.2"
SR = 16000
DURATION = 5.0
N_MFCC = 13
NUM_FRAMES = 10

In [ ]:
def get_video_paths_labels(dataset_dir):
    all_paths_labels = []
    # Gerçek ses: hem RealVideo-RealAudio hem de FakeVideo-RealAudio (ses gerçek)
    real_audio_dirs = ["RealVideo-RealAudio", "FakeVideo-RealAudio"]
    # Sahte ses: RealVideo-FakeAudio, FakeVideo-FakeAudio
    fake_audio_dirs = ["RealVideo-FakeAudio", "FakeVideo-FakeAudio"]

    for r_dir in real_audio_dirs:
        path = os.path.join(dataset_dir, r_dir)
        videos = glob.glob(os.path.join(path, "**", "*.mp4"), recursive=True)
        for rv in videos:
            all_paths_labels.append((rv, 0))
    for f_dir in fake_audio_dirs:
        path = os.path.join(dataset_dir, f_dir)
        videos = glob.glob(os.path.join(path, "**", "*.mp4"), recursive=True)
        for fv in videos:
            all_paths_labels.append((fv, 1))
    return all_paths_labels

In [ ]:
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True,
                                  max_num_faces=1,
                                  refine_landmarks=True,
                                  min_detection_confidence=0.5)

In [ ]:
LIPS = [
    61, 185, 40, 39, 37,   0, 267, 269, 270, 409, 291,
    146, 91, 181, 84, 17, 314, 405, 321, 375,  308,
    78, 95, 88, 178, 87, 14, 317, 402, 318, 324,
    308, 191, 80, 81, 82, 13, 312, 311, 310, 415
]


In [ ]:
def extract_lip_region(frame):
    # frame: BGR formatında, OpenCV okur
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)
    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0]
        h, w, _ = frame.shape
        xs = [landmark.x * w for idx, landmark in enumerate(landmarks.landmark) if idx in LIPS]
        ys = [landmark.y * h for idx, landmark in enumerate(landmarks.landmark) if idx in LIPS]
        if xs and ys:
            x_min, x_max = int(min(xs)), int(max(xs))
            y_min, y_max = int(min(ys)), int(max(ys))
            # Kırpma yap, biraz margin ekleyebiliriz
            margin = 5
            x_min = max(x_min - margin, 0)
            y_min = max(y_min - margin, 0)
            x_max = min(x_max + margin, w)
            y_max = min(y_max + margin, h)
            lip_region = frame[y_min:y_max, x_min:x_max]
            lip_region = cv2.cvtColor(lip_region, cv2.COLOR_BGR2GRAY)
            lip_region = cv2.resize(lip_region, (64, 64))
            lip_region = np.expand_dims(lip_region, axis=-1)  # (64,64,1)
            return lip_region
    return None


In [ ]:
def extract_audio_mfcc(video_path, sr=16000, duration=3.0, n_mfcc=13):
    temp_audio = "temp_audio.wav"
    command = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-ac", "1",
        "-ar", str(sr),
        temp_audio
    ]
    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    y, _ = librosa.load(temp_audio, sr=sr, duration=duration)
    target_length = int(sr * duration)
    if len(y) < target_length:
        y = np.concatenate([y, np.zeros(target_length - len(y))])
    else:
        y = y[:target_length]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc = np.expand_dims(mfcc, axis=-1)  # (n_mfcc, time_frames, 1)
    return mfcc

In [ ]:
def prepare_av_sample(video_path, sr=16000, duration=3.0, n_mfcc=13, num_frames=10):
    cap = cv2.VideoCapture(video_path)

    # Videonun 3 saniyelik kısmı için fps * duration kadar kare hedefleniyor
    fps = cap.get(cv2.CAP_PROP_FPS)
    num_target_frames = int(fps * duration) if fps > 0 else 0
    frame_indices = np.linspace(0, max(0, num_target_frames - 1), num_frames, dtype=int)

    current_frame = 0
    selected_frames = []

    while cap.isOpened() and len(selected_frames) < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if current_frame in frame_indices:
            lip = extract_lip_region(frame)  # Dudak bölgesi çıkarma fonksiyonunuz
            if lip is not None:
                selected_frames.append(lip)
        current_frame += 1
    cap.release()

    # Eğer hiçbir dudak karesi toplanamadıysa (liste boşsa) None döndür
    if len(selected_frames) == 0:
        return None, None

    # Toplanan kare sayısı hedefin altındaysa, son kareyi tekrar ederek pad et
    if len(selected_frames) < num_frames:
        for _ in range(num_frames - len(selected_frames)):
            selected_frames.append(selected_frames[-1])

    visual_sequence = np.array(selected_frames)  # (num_frames, 64, 64, 1)

    # Ses için MFCC hesapla
    mfcc = extract_audio_mfcc(video_path, sr=sr, duration=duration, n_mfcc=n_mfcc)
    # Eğer mfcc hiç hesaplanamadıysa veya sıfır boyutlu ise None döndür
    if mfcc is None or mfcc.shape[1] == 0:
        return None, None

    # mfcc: (n_mfcc, T, 1) – T, MFCC zaman çerçeveleri sayısı
    T = mfcc.shape[1]
    # Zaman eksenini num_frames segmente bölelim
    mfcc_segments = np.array_split(mfcc, num_frames, axis=1)
    audio_sequence = np.array([segment.mean(axis=1) for segment in mfcc_segments])  # (num_frames, n_mfcc, 1)
    audio_sequence = np.squeeze(audio_sequence, axis=-1)  # (num_frames, n_mfcc)
    audio_sequence = np.expand_dims(audio_sequence, axis=-1)  # (num_frames, n_mfcc, 1)

    return visual_sequence, audio_sequence


In [ ]:
def prepare_av_dataset(paths_labels, sr=16000, duration=3.0, n_mfcc=13, num_frames=10):
    X_visual = []
    X_audio = []
    y = []
    for video_path, label in paths_labels:
        visual_seq, audio_seq = prepare_av_sample(video_path, sr=sr, duration=duration, n_mfcc=n_mfcc, num_frames=num_frames)
        if visual_seq is None or audio_seq is None:
            continue
        X_visual.append(visual_seq)
        X_audio.append(audio_seq)
        y.append(label)
    return np.array(X_visual), np.array(X_audio), np.array(y)

In [ ]:
def build_visual_branch(input_shape=(10, 64, 64, 1)):
    visual_input = layers.Input(shape=input_shape)
    # Zaman boyutundaki her kare için CNN uygulayalım (TimeDistributed)
    x = layers.TimeDistributed(layers.Conv2D(32, (3,3), activation='relu', padding='same'))(visual_input)
    x = layers.TimeDistributed(layers.MaxPooling2D((2,2)))(x)
    x = layers.TimeDistributed(layers.Conv2D(64, (3,3), activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)
    # Şimdi LSTM ile zaman ilişkisini modelleyelim
    x = layers.LSTM(128)(x)
    visual_features = layers.Dense(128, activation='relu')(x)
    return models.Model(visual_input, visual_features, name="visual_branch")

# Ses branch: Zaman serisindeki MFCC segmentlerini işleyen LSTM + CNN
def build_audio_branch(input_shape=(10, 13, 1)):
    audio_input = layers.Input(shape=input_shape)
    # Zaman boyutundaki her MFCC segmenti için küçük CNN
    x = layers.TimeDistributed(layers.Conv1D(16, 3, activation='relu', padding='same'))(audio_input)
    x = layers.TimeDistributed(layers.MaxPooling1D(2))(x)
    x = layers.TimeDistributed(layers.Conv1D(32, 3, activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling1D())(x)
    x = layers.LSTM(128)(x)
    audio_features = layers.Dense(128, activation='relu')(x)
    return models.Model(audio_input, audio_features, name="audio_branch")

# Audio-Visual Model: İki branch'in çıktısını birleştirip senkronizasyon ve deepfake tespiti yapar
def build_av_model(visual_input_shape=(10, 64, 64, 1), audio_input_shape=(10, 13, 1)):
    visual_branch = build_visual_branch(visual_input_shape)
    audio_branch = build_audio_branch(audio_input_shape)
    combined = layers.concatenate([visual_branch.output, audio_branch.output])
    x = layers.Dense(64, activation='relu')(combined)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=[visual_branch.input, audio_branch.input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
paths_labels = get_video_paths_labels(DATASET_DIR)
print("Toplam örnek sayısı:", len(paths_labels))

Toplam örnek sayısı: 21544


In [ ]:
train_paths, test_paths = train_test_split(paths_labels, test_size=0.2, random_state=42, stratify=[lbl for _, lbl in paths_labels])
print("Eğitim örnek sayısı:", len(train_paths))
print("Test örnek sayısı:", len(test_paths))

print("Veri hazırlanıyor (audio-visual)...")
X_visual_train, X_audio_train, y_train = prepare_av_dataset(train_paths, sr=SR, duration=DURATION, n_mfcc=N_MFCC, num_frames=NUM_FRAMES)
X_visual_test, X_audio_test, y_test = prepare_av_dataset(test_paths, sr=SR, duration=DURATION, n_mfcc=N_MFCC, num_frames=NUM_FRAMES)
print("Eğitim görsel verisi şekli:", X_visual_train.shape)
print("Eğitim ses verisi şekli:", X_audio_train.shape)
print("Test görsel verisi şekli:", X_visual_test.shape)
print("Test ses verisi şekli:", X_audio_test.shape)

Eğitim örnek sayısı: 17235
Test örnek sayısı: 4309
Veri hazırlanıyor (audio-visual)...
Eğitim görsel verisi şekli: (17233, 10, 64, 64, 1)
Eğitim ses verisi şekli: (17233, 10, 13, 1)
Test görsel verisi şekli: (4308, 10, 64, 64, 1)
Test ses verisi şekli: (4308, 10, 13, 1)


In [ ]:
# Modeli oluştur
av_model = build_av_model(visual_input_shape=(NUM_FRAMES, 64, 64, 1), audio_input_shape=(NUM_FRAMES, N_MFCC, 1))
av_model.summary()

# Modeli eğit
history = av_model.fit([X_visual_train, X_audio_train], y_train, epochs=10, batch_size=16, validation_split=0.2)

# Modeli test et
test_loss, test_acc = av_model.evaluate([X_visual_test, X_audio_test], y_test)
print("Test Loss:", test_loss, "Test Accuracy:", test_acc)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 10, 64, 64, 1)  │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_1             │ (None, 10, 13, 1)      │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed          │ (None, 10, 64, 64, 32) │            320 │ input_layer[0][0]      │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_4        │ (None, 10, 13, 16)     │             64 │ input_layer_1[0][0]    │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_1        │ (None, 10, 32, 32, 32) │              0 │ time_distributed[0][0] │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_5        │ (None, 10, 6, 16)      │              0 │ time_distributed_4[0]… │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_2        │ (None, 10, 32, 32, 64) │         18,496 │ time_distributed_1[0]… │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_6        │ (None, 10, 6, 32)      │          1,568 │ time_distributed_5[0]… │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_3        │ (None, 10, 64)         │              0 │ time_distributed_2[0]… │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_7        │ (None, 10, 32)         │              0 │ time_distributed_6[0]… │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm (LSTM)               │ (None, 128)            │         98,816 │ time_distributed_3[0]… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_1 (LSTM)             │ (None, 128)            │         82,432 │ time_distributed_7[0]… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 128)            │         16,512 │ lstm[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 128)            │         16,512 │ lstm_1[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 256)            │              0 │ dense[0][0],           │
│                      

 Total params: 251,233 (981.38 KB)

 Trainable params: 251,233 (981.38 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 59s 54ms/step - accuracy: 0.6972 - loss: 0.5371 - val_accuracy: 0.8091 - val_loss: 0.3926
Epoch 2/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 78s 54ms/step - accuracy: 0.8157 - loss: 0.3806 - val_accuracy: 0.8578 - val_loss: 0.3129
Epoch 3/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 80s 51ms/step - accuracy: 0.8633 - loss: 0.3069 - val_accuracy: 0.9017 - val_loss: 0.2233
Epoch 4/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 47s 54ms/step - accuracy: 0.9192 - loss: 0.1895 - val_accuracy: 0.9518 - val_loss: 0.1167
Epoch 5/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 79s 50ms/step - accuracy: 0.9561 - loss: 0.1097 - val_accuracy: 0.9768 - val_loss: 0.0662
Epoch 6/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 85s 54ms/step - accuracy: 0.9776 - loss: 0.0622 - val_accuracy: 0.9861 - val_loss: 0.0394
Epoch 7/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 44s 51ms/step - accuracy: 0.9862 - loss: 0.0395 - val_accuracy: 0.9867 - val_loss: 0.0406
Epoch 8/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 83s 51ms/step - accuracy: 0.9836 - loss: 0.0449 - 

In [ ]:
test_loss, test_acc = av_model.evaluate([X_visual_test, X_audio_test], y_test)
print("Test Loss:", test_loss, "Test Accuracy:", test_acc)

135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9838 - loss: 0.0486
Test Loss: 0.03916839137673378 Test Accuracy: 0.9870009422302246


In [ ]:
av_model.save("cnn_lip_model.h5")
import shutil
shutil.move("cnn_lip_model.h5", "/content/drive/My Drive/cnn_lip_model.h5")

'/content/drive/My Drive/cnn_lip_model.h5'